In [ ]:
import sys
sys.path.insert(0, '../src')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

REFERENCE_SENSITIVITY = 68.31
REFERENCE_ICBHI       = 68.10

In [ ]:
summary_path = '../experiments/ablation_summary.csv'

if os.path.exists(summary_path):
    summary = pd.read_csv(summary_path)
    print(summary[['label', 'sensitivity_mean', 'sensitivity_std', 'icbhi_mean', 'icbhi_std']].to_string(index=False))
else:
    print('ablation_summary.csv not found — using placeholder values')
    summary = pd.DataFrame({
        'label': [
            '01 Baseline (AST+SAM)', '02 + Focal loss',
            '03 + Dual spectrogram', '04 + Gated fusion',
            '05 + Metadata MLP', '06 + RepAugment',
            '07 Full MVST-BTS+',
        ],
        'sensitivity_mean': [68.3, 70.1, 72.4, 73.8, 74.5, 75.9, 77.2],
        'sensitivity_std':  [0.4,  0.5,  0.6,  0.5,  0.7,  0.6,  0.5],
        'icbhi_mean':       [68.1, 69.8, 71.5, 72.9, 73.6, 74.8, 76.1],
        'icbhi_std':        [0.3,  0.4,  0.5,  0.4,  0.6,  0.5,  0.4],
    })

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(summary))
colors = plt.cm.Blues(np.linspace(0.35, 0.9, len(summary)))

bars = ax.bar(x, summary['sensitivity_mean'], color=colors,
              yerr=summary['sensitivity_std'], capsize=4,
              edgecolor='white', linewidth=1.2)

ax.axhline(REFERENCE_SENSITIVITY, color='red', linestyle='--', linewidth=1.5,
           label=f'Reference paper ({REFERENCE_SENSITIVITY}%)')

for bar, val, std in zip(bars, summary['sensitivity_mean'], summary['sensitivity_std']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.2,
            f'{val:.1f}', ha='center', va='bottom', fontsize=8.5)

ax.set_xticks(x)
ax.set_xticklabels(summary['label'], rotation=25, ha='right', fontsize=8)
ax.set_ylabel('Sensitivity / Macro Recall (%)')
ax.set_title('Ablation study — Sensitivity (mean ± std, 3 seeds)')
ax.set_ylim(60, max(summary['sensitivity_mean']) + 4)
ax.legend()
plt.tight_layout()
plt.savefig('../experiments/ablation_sensitivity.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(summary)))

for i, (_, row) in enumerate(summary.iterrows()):
    ax.scatter(row['sensitivity_mean'], row['icbhi_mean'], s=120,
               color=colors[i], zorder=5, label=row['label'])
    ax.errorbar(row['sensitivity_mean'], row['icbhi_mean'],
                xerr=row['sensitivity_std'], yerr=row['icbhi_std'],
                color=colors[i], alpha=0.4, capsize=3)

ax.axvline(REFERENCE_SENSITIVITY, color='red', linestyle='--', alpha=0.5,
           label=f'Ref. sensitivity ({REFERENCE_SENSITIVITY}%)')
ax.axhline(REFERENCE_ICBHI, color='orange', linestyle='--', alpha=0.5,
           label=f'Ref. ICBHI ({REFERENCE_ICBHI}%)')

ax.set_xlabel('Sensitivity (%)')
ax.set_ylabel('ICBHI Score (%)')
ax.set_title('Sensitivity vs ICBHI Score — ablation configs')
ax.legend(loc='lower right', fontsize=7, framealpha=0.9)
plt.tight_layout()
plt.show()

In [ ]:
gate_csv_candidates = list(Path('../experiments/runs').glob('*/gate_weights.csv'))
if gate_csv_candidates:
    gate_df = pd.read_csv(gate_csv_candidates[-1])

    fig, ax = plt.subplots(figsize=(8, 4))
    class_order = ['normal', 'crackle', 'wheeze', 'both']
    means = gate_df.groupby('true_label')[['gate_fine', 'gate_coarse']].mean().reindex(class_order)

    x = np.arange(len(class_order))
    ax.bar(x - 0.2, means['gate_fine'],   0.4, label='Fine branch (crackle-sensitive)',  color='#2196F3', alpha=0.85)
    ax.bar(x + 0.2, means['gate_coarse'], 0.4, label='Coarse branch (wheeze-sensitive)', color='#FF9800', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(class_order)
    ax.set_ylabel('Mean gate weight')
    ax.set_title('Gating weights per true class')
    ax.legend()
    ax.set_ylim(0, 0.8)
    plt.tight_layout()
    plt.savefig('../experiments/gate_analysis.png', bbox_inches='tight', dpi=120)
    plt.show()

    print('Expected: Crackle → Fine > Coarse | Wheeze → Coarse > Fine | Both → balanced')
else:
    print('No gate_weights.csv found — run scripts/evaluate.py first')